In [1]:
import enum
import json
import os
from copy import deepcopy

import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
from sklearn.metrics import f1_score
from timm.utils import ModelEmaV2
from torch.nn.functional import softmax
from torch.optim import AdamW
from torch.utils.data import DataLoader
from tqdm import tqdm

from internal.data_types import HistologyDataset
from internal.nn.dual_path_net import DualPathNet
from internal.nn.mixup_cutmix_wrapper import MixupCutmixWrapper
from internal.nn.model import train_one_epoch, validate
from internal.nn.test_time_augmentation import apply_mask_multicrop_tta
from internal.nn.test_time_augmentation import apply_tta
from internal.nn.test_time_augmentation import apply_tta_4ch_safe
from internal.nn.weighted_random_sampler import make_weighted_sampler
from internal.persistence_manager import PersistenceManager

data = PersistenceManager.load_dataset()
test_df = data.test_df
train_df = data.train_df
train_transforms = data.train_transforms
val_test_transforms = data.val_test_transforms
idx2label = data.idx2label

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
cuda_is_available = torch.cuda.is_available()
print(f'Using device: {device}')

Arrays and scalers loaded successfully from: /home/andre/university/AN2DL-Challenge-2/notebooks/processed/dataset.joblib
Using device: cuda


In [2]:
def get_classifier_module(model: nn.Module):
    # Common names in timm models
    for name in ["classifier", "fc", "head"]:
        if hasattr(model, name):
            return getattr(model, name), name
    # Fallback: assume there is a single linear at the very end
    last_linear = None
    for m in reversed(list(model.modules())):
        if isinstance(m, nn.Linear):
            last_linear = m
            break
    if last_linear is None:
        raise RuntimeError("Could not find classifier layer in model.")
    return last_linear, None

In [3]:
class PreTrainedArchitectures(enum.Enum):
    EFFICIENTNETV2_S = "tf_efficientnetv2_s.in21k"
    EFFICIENTNET_B1_NS = "tf_efficientnet_b1.ns_jft_in1k"
    CONVNEXT_TINY = "convnext_tiny"
    EFFICIENTNET_B0 = "efficientnet_b0"
    EFFICIENTNET_B1 = "efficientnet_b1"

MODEL_TO_USE: PreTrainedArchitectures = PreTrainedArchitectures.EFFICIENTNET_B1_NS

In [4]:
best_f1_per_fold: dict[int, int] = {}
N_WORKERS = os.cpu_count() // 2 if os.cpu_count() else 4
N_CLASSES = 4  # number of classes in the dataset (labels)
EMA_DECAY = 0.999
USE_DUAL_PATH_NET = False

# efficientnet_b0 / efficientnet_b1

In [5]:
def create_efficientnet_b0_model(pretrained: bool = True) -> nn.Module:
    model = timm.create_model(
        PRETRAINED_MODEL,
        pretrained=pretrained,
        num_classes=N_CLASSES,
        in_chans=4,
        drop_rate=0.3,          # Dropout
        drop_path_rate=0.1      # Stochastic depth
    ).to(device)
    return model

def freeze_all(model: nn.Module):
    for p in model.parameters():
        p.requires_grad = False

def unfreeze_last_two_blocks_and_head(model: nn.Module):
    """
    For EfficientNet from timm: unfreeze last 2 blocks + classifier head.
    """
    freeze_all(model)

    # Last 2 conv blocks
    if hasattr(model, "blocks"):
        for blk in model.blocks[-2:]:
            for p in blk.parameters():
                p.requires_grad = True

    # Classifier head
    clf_module, _ = get_classifier_module(model)
    for p in clf_module.parameters():
        p.requires_grad = True


if MODEL_TO_USE == PreTrainedArchitectures.EFFICIENTNET_B0 or MODEL_TO_USE == PreTrainedArchitectures.EFFICIENTNET_B1:
    N_FOLDS = data.num_K_folds
    IMAGE_SIZE = data.image_size
    BATCH_SIZE = 4
    GRAD_ACCUM_STEPS = 2
    PRETRAINED_MODEL = MODEL_TO_USE.value
    N_CLASSES = 4 # number of classes in the dataset (labels)
    N_WORKERS = os.cpu_count() // 2 if os.cpu_count() else 4
    EPOCHS = 25
    LR = 3e-4
    PREFIX = "effb0" if MODEL_TO_USE == PreTrainedArchitectures.EFFICIENTNET_B0 else "effb1"

    for fold in range(N_FOLDS):
        print(f"\n========== Fold {fold} ==========")

        train_df_split = train_df[train_df["fold"] != fold].reset_index(drop=True)
        val_df_split   = train_df[train_df["fold"] == fold].reset_index(drop=True)

        train_dataset = HistologyDataset(
            df=train_df_split,
            image_size=IMAGE_SIZE,
            is_train=True,  # Augmentations applied
            use_mask_crop=True
        )
        val_dataset = HistologyDataset(
            df=val_df_split,
            image_size=IMAGE_SIZE,
            is_train=False, # Disable augmentations
            use_mask_crop=True
        )

        sampler = make_weighted_sampler(train_df_split)
        train_loader = DataLoader(
            train_dataset,
            batch_size=BATCH_SIZE,
            sampler=sampler,
            num_workers=N_WORKERS,
            pin_memory=cuda_is_available
        )
        val_loader   = DataLoader(
            val_dataset,
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=N_WORKERS,
            pin_memory=cuda_is_available
        )
        # ---- create model + unfreeze last 2 blocks + head ----
        model = create_efficientnet_b0_model(pretrained=True)
        unfreeze_last_two_blocks_and_head(model)

        # ---- loss, optimizer, scheduler ----
        class_counts_np = train_df["label_idx"].value_counts().sort_index().values
        class_counts = torch.tensor(class_counts_np, dtype=torch.float32)
        class_weights = (class_counts.sum() / class_counts)
        class_weights = class_weights / class_weights.mean()

        criterion = nn.CrossEntropyLoss(
            weight=class_weights.to(device),
            label_smoothing=0.1
        )

        optimizer = AdamW(
            [p for p in model.parameters() if p.requires_grad],
            lr=LR,
            weight_decay=1e-4
        )
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=EPOCHS
        )

        # ---- training loop ----
        best_f1 = 0.0
        best_state = None
        mixup_fn = MixupCutmixWrapper(
            alpha=0.4,       # mixup/cutmix Beta distribution
            mixup_prob=0.4,  # 40% of batches => mixup
            cutmix_prob=0.2  # 20% of batches => cutmix
        )

        for epoch in range(1, EPOCHS + 1):
            print(f"\nEpoch {epoch}/{EPOCHS}")
            train_loss, train_acc, train_f1 = train_one_epoch(
                model, train_loader, optimizer, criterion, device, grad_accum_steps=GRAD_ACCUM_STEPS, mixup_fn=mixup_fn
            )
            val_loss, val_acc, val_f1 = validate(
                model, val_loader, criterion, device
            )
            scheduler.step()

            print(
                f"Train  loss={train_loss:.4f} acc={train_acc:.4f} f1={train_f1:.4f} | "
                f"Val loss={val_loss:.4f} acc={val_acc:.4f} f1={val_f1:.4f}"
            )

            if val_f1 > best_f1:
                best_f1 = val_f1
                best_state = deepcopy(model.state_dict())
                torch.save(
                    best_state,
                    f"best_effb0_fold{fold}_f1_{val_f1:.4f}.pth"
                )
                print(f"  🔥 New best F1: {best_f1:.4f} – model saved.")

        # restore best weights for this fold
        if best_state is not None:
            model.load_state_dict(best_state)
            print(f"Restored best weights for fold {fold} (F1={best_f1:.4f})")

        # save final model for inference
        torch.save(model.state_dict(), f"effb0_fold{fold}.pth")

        # record best F1 for this fold
        best_f1_per_fold[fold] = best_f1

# tf_efficientnet_b1_ns

In [6]:
def create_efficientnet_b1_ns_model(pretrained: bool = True) -> nn.Module:
    model = (
        DualPathNet(
            backbone_name=MODEL_TO_USE.value,
            num_classes=N_CLASSES,
            pretrained=pretrained,
            mask_feat_dim=128,
            drop_rate=0.4,       # stronger dropout than B0
            drop_path_rate=0.15  # stochastic depth
        )
        if USE_DUAL_PATH_NET
        else
        timm.create_model(
            MODEL_TO_USE.value,           # tf_efficientnet_b1_ns
            pretrained=pretrained,
            num_classes=N_CLASSES,
            in_chans=4,                   # 3 RGB + 1 mask
            drop_rate=0.4,
            drop_path_rate=0.15
        )
    ).to(device)
    return model

def freeze_all(model: nn.Module):
    for p in model.parameters():
        p.requires_grad = False

def unfreeze_all(model: nn.Module):
    for p in model.parameters():
        p.requires_grad = True

def unfreeze_last_two_blocks_and_head(model: nn.Module):
    """
    Freeze earlier EfficientNet blocks, unfreeze the last two + head.
    Works for timm tf_efficientnet_b* models.
    """
    # 1) Freeze everything by default
    for p in model.parameters():
        p.requires_grad = False

    # 2) Unfreeze last two blocks
    # model.blocks is a nn.Sequential
    num_blocks = len(model.blocks)
    for idx in range(num_blocks - 2, num_blocks):
        for p in model.blocks[idx].parameters():
            p.requires_grad = True

    # 3) Unfreeze conv_head + bn2 + classifier
    for p in model.conv_head.parameters():
        p.requires_grad = True
    for p in model.bn2.parameters():
        p.requires_grad = True
    for p in model.classifier.parameters():
        p.requires_grad = True

def unfreeze_last_block_and_head(model: nn.Module, n_blocks: int = 2):
    # 1) freeze everything
    for p in model.parameters():
        p.requires_grad = False

    # 2) unfreeze last n_blocks of the RGB backbone
    # efficientnet-style timm models have .blocks
    if hasattr(model.rgb_backbone, "blocks"):
        for block in model.rgb_backbone.blocks[-n_blocks:]:
            for p in block.parameters():
                p.requires_grad = True
    else:
        # fallback: unfreeze entire backbone if the structure is different
        for p in model.rgb_backbone.parameters():
            p.requires_grad = True

    # 3) always train mask branch + fusion classifier
    for p in model.mask_branch.parameters():
        p.requires_grad = True
    for p in model.mask_fc.parameters():
        p.requires_grad = True
    for p in model.classifier.parameters():
        p.requires_grad = True


def unfreeze_last_stage_and_head(model: nn.Module):
    """
    EfficientNet B1-NS recommended fine-tuning strategy:
    - Freeze all early MBConv stages
    - Unfreeze the last MBConv stage (stage 6)
    - Unfreeze conv_head + bn2 + classifier
    """

    # Freeze everything first
    for p in model.parameters():
        p.requires_grad = False

    # ---- Unfreeze last stage (stage 6) ----
    # EfficientNet blocks are sequential but grouped in stages.
    # B1 layout roughly:
    #   Stage0: stem
    #   Stage1: blocks[0]
    #   Stage2: blocks[1:3]
    #   Stage3: blocks[3:5]
    #   Stage4: blocks[5:8]
    #   Stage5: blocks[8:11]
    #   Stage6: blocks[11:15]  <-- last stage
    last_stage_start = len(model.blocks) - 4  # 4 blocks in last stage (B1)
    for idx in range(last_stage_start, len(model.blocks)):
        for p in model.blocks[idx].parameters():
            p.requires_grad = True

    # ---- Unfreeze head ----
    for p in model.conv_head.parameters():
        p.requires_grad = True
    for p in model.bn2.parameters():
        p.requires_grad = True
    for p in model.classifier.parameters():
        p.requires_grad = True

if MODEL_TO_USE == PreTrainedArchitectures.EFFICIENTNET_B1_NS:
    N_FOLDS = data.num_K_folds
    IMAGE_SIZE = data.image_size
    BATCH_SIZE = 4
    GRAD_ACCUM_STEPS = 2
    PRETRAINED_MODEL = MODEL_TO_USE.value
    N_CLASSES = 4 # number of classes in the dataset (labels)
    N_WORKERS = os.cpu_count() // 2 if os.cpu_count() else 4
    EPOCHS = 50
    LR = 1e-4
    WEIGHT_DECAY = 1e-4
    MIXUP_CUTMIX_ALPHA = 0.2
    MIXUP_PROB = 0.0
    CUTMIX_PROB = 0.0
    PREFIX = "tf_effb1_ns"
    USE_EMA = True
    USE_MIXUP_CUTMIX = False
    USE_FREEZE_TECHNIQUE = True
    PATIENCE = 10

    for fold in range(N_FOLDS):
        print(f"\n========== Fold {fold} ==========")

        train_df_split = train_df[train_df["fold"] != fold].reset_index(drop=True)
        val_df_split   = train_df[train_df["fold"] == fold].reset_index(drop=True)

        train_dataset = HistologyDataset(
            df=train_df_split,
            image_size=IMAGE_SIZE,
            is_train=True,          # Augmentations applied
            use_mask_crop=True,
            apply_artifact_augs=False,
            apply_random_erasing=False,
            patch_mode=True,       # Use patch-based training
            patches_per_image=3,    # Add 3 random patches per image
            patch_size=384          # 384x384 patches
        )
        val_dataset = HistologyDataset(
            df=val_df_split,
            image_size=IMAGE_SIZE,
            is_train=False, # Disable augmentations
            use_mask_crop=True,
            apply_artifact_augs=False,
            apply_random_erasing=False,
            patch_mode=False
        )

        sampler = make_weighted_sampler(train_df_split)
        train_loader = DataLoader(
            train_dataset,
            batch_size=BATCH_SIZE,
            sampler=sampler,
            num_workers=N_WORKERS,
            pin_memory=cuda_is_available
        )
        val_loader   = DataLoader(
            val_dataset,
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=N_WORKERS,
            pin_memory=cuda_is_available
        )
        # ---- create model + unfreeze last 2 blocks + head ----
        model = create_efficientnet_b1_ns_model(pretrained=True)
        unfreeze_all(model)
        if USE_FREEZE_TECHNIQUE:
            if USE_DUAL_PATH_NET:
                unfreeze_last_block_and_head(model)
            else:
                unfreeze_last_two_blocks_and_head(model)

        # --- EMA ---
        ema_model = ModelEmaV2(model, decay=EMA_DECAY, device=device) if USE_EMA else None

        # ---- loss, optimizer, scheduler ----
        # class_counts_np = train_df_split["label_idx"].value_counts().sort_index().values
        # print("Class counts:", class_counts_np)
        # class_counts = torch.tensor(class_counts_np, dtype=torch.float32)
        # class_weights = (class_counts.sum() / class_counts)
        # class_weights = (class_counts.sum() / class_counts).sqrt()
        # class_weights = class_weights / class_weights.mean()
        # criterion = nn.CrossEntropyLoss()

        class_counts = train_df_split["label_idx"].value_counts().sort_index().values
        print("Class counts:", class_counts)
        w = torch.tensor(class_counts.sum() / class_counts, dtype=torch.float32)
        w = (w / w.mean()).to(device)
        criterion = nn.CrossEntropyLoss(weight=w, label_smoothing=0.05)


        optimizer = AdamW(
            [p for p in model.parameters() if p.requires_grad],
            lr=LR,
            weight_decay=WEIGHT_DECAY
        )
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=EPOCHS
        )

        # ---- training loop ----
        best_f1 = 0.0
        best_epoch = 0
        best_state = None
        mixup_fn = MixupCutmixWrapper(
            alpha=MIXUP_CUTMIX_ALPHA,   # mixup/cutmix Beta distribution
            mixup_prob=MIXUP_PROB,      # x% of batches => mixup
            cutmix_prob=CUTMIX_PROB     # x% of batches => cutmix
        ) if USE_MIXUP_CUTMIX else None

        for epoch in range(1, EPOCHS + 1):
            print(f"\nEpoch {epoch}/{EPOCHS} - Fold {fold}/{N_FOLDS-1} - Best F1: {best_f1:.4f} at epoch {best_epoch}")
            if epoch == 6:
                unfreeze_all(model)             # or unfreeze_last_stage_and_head(model)
                for g in optimizer.param_groups:
                    g["lr"] *= 0.3              # lower LR when you unfreeze more


            train_loss, train_acc, train_f1 = train_one_epoch(
                model,
                train_loader,
                optimizer,
                criterion,
                device,
                grad_accum_steps=GRAD_ACCUM_STEPS,
                mixup_fn=mixup_fn,
                ema_model=ema_model
            )

            val_loss, val_acc, val_f1 = validate(
                ema_model.module if ema_model else model,   # use EMA weights for validation
                val_loader,
                criterion,
                device,
                print_report=True
            )
            scheduler.step()

            print(
                f"Train  loss={train_loss:.4f} acc={train_acc:.4f} f1={train_f1:.4f} | "
                f"Val loss={val_loss:.4f} acc={val_acc:.4f} f1={val_f1:.4f}"
            )

            if val_f1 > best_f1:
                best_f1 = val_f1
                best_epoch = epoch
                best_state = deepcopy(
                    ema_model.module.state_dict()
                    if ema_model else model.state_dict()
                )
                torch.save(
                    best_state,
                    f"best_{PREFIX}_fold{fold}_f1_{val_f1:.4f}.pth"
                )
                print(f"  🔥 New best F1: {best_f1:.4f} – model saved.")
            # if the model overfits too much, we can stop early
            if epoch - best_epoch >= PATIENCE:
                print("Early stopping due to no improvement in 7 epochs.")
                break

        # restore best EMA weights for this fold
        if best_state is not None:
            if USE_EMA:
                ema_model.module.load_state_dict(best_state)
            else:
                model.load_state_dict(best_state)
            print(f"Restored best weights for fold {fold} (F1={best_f1:.4f})")

        # save final model for inference
        if USE_EMA:
            torch.save(ema_model.module.state_dict(), f"{PREFIX}_fold{fold}.pth")
        else:
            torch.save(model.state_dict(), f"{PREFIX}_fold{fold}.pth")

        # record best F1 for this fold
        best_f1_per_fold[fold] = best_f1


========== Fold 0 ==========
Class counts: [163 126 120  55]

Epoch 1/50 - Fold 0/4 - Best F1: 0.0000 at epoch 0


    t_loss=2.2242 | F1(macro)=0.2572 | Acc=0.2823


Confusion matrix:
 [[ 0 19  0 22]
 [ 0 15  1 16]
 [ 0 14  1 15]
 [ 0  8  0  6]]

Per-class report (VAL):
              precision    recall  f1-score   support

           0      0.000     0.000     0.000        41
           1      0.268     0.469     0.341        32
           2      0.500     0.033     0.062        30
           3      0.102     0.429     0.164        14

    accuracy                          0.188       117
   macro avg      0.217     0.233     0.142       117
weighted avg      0.214     0.188     0.129       117

Pred distribution: [ 0 56  2 59]
True distribution: [41 32 30 14]
Train  loss=2.2242 acc=0.2823 f1=0.2572 | Val loss=2.0317 acc=0.1880 f1=0.1419
  🔥 New best F1: 0.1419 – model saved.

Epoch 2/50 - Fold 0/4 - Best F1: 0.1419 at epoch 1


    t_loss=1.9176 | F1(macro)=0.2869 | Acc=0.3297


Confusion matrix:
 [[ 7 32  0  2]
 [ 6 25  1  0]
 [ 6 20  4  0]
 [ 2 11  1  0]]

Per-class report (VAL):
              precision    recall  f1-score   support

           0      0.333     0.171     0.226        41
           1      0.284     0.781     0.417        32
           2      0.667     0.133     0.222        30
           3      0.000     0.000     0.000        14

    accuracy                          0.308       117
   macro avg      0.321     0.271     0.216       117
weighted avg      0.365     0.308     0.250       117

Pred distribution: [21 88  6  2]
True distribution: [41 32 30 14]
Train  loss=1.9176 acc=0.3297 f1=0.2869 | Val loss=2.2607 acc=0.3077 f1=0.2162
  🔥 New best F1: 0.2162 – model saved.

Epoch 3/50 - Fold 0/4 - Best F1: 0.2162 at epoch 2


...

Confusion matrix:
 [[17 16  5  2]
 [18 13  1  0]
 [15 14  1  0]
 [ 7  5  2  0]]

Per-class report (VAL):
              precision    recall  f1-score   support

           0      0.298     0.425     0.351        40
           1      0.271     0.406     0.325        32
           2      0.111     0.033     0.051        30
           3      0.000     0.000     0.000        14

    accuracy                          0.267       116
   macro avg      0.170     0.216     0.182       116
weighted avg      0.206     0.267     0.224       116

Pred distribution: [57 48  9  2]
True distribution: [40 32 30 14]
Train  loss=1.5070 acc=0.3376 f1=0.3319 | Val loss=1.9599 acc=0.2672 f1=0.1817

Epoch 15/50 - Fold 4/4 - Best F1: 0.2337 at epoch 6


    t_loss=1.5053 | F1(macro)=0.3795 | Acc=0.3957


Confusion matrix:
 [[16 18  4  2]
 [17 13  2  0]
 [15 14  1  0]
 [ 8  5  1  0]]

Per-class report (VAL):
              precision    recall  f1-score   support

           0      0.286     0.400     0.333        40
           1      0.260     0.406     0.317        32
           2      0.125     0.033     0.053        30
           3      0.000     0.000     0.000        14

    accuracy                          0.259       116
   macro avg      0.168     0.210     0.176       116
weighted avg      0.203     0.259     0.216       116

Pred distribution: [56 50  8  2]
True distribution: [40 32 30 14]
Train  loss=1.5053 acc=0.3957 f1=0.3795 | Val loss=1.9333 acc=0.2586 f1=0.1758

Epoch 16/50 - Fold 4/4 - Best F1: 0.2337 at epoch 6


    t_loss=1.4437 | F1(macro)=0.3646 | Acc=0.3914


Confusion matrix:
 [[16 17  5  2]
 [16 15  1  0]
 [13 16  1  0]
 [ 6  7  1  0]]

Per-class report (VAL):
              precision    recall  f1-score   support

           0      0.314     0.400     0.352        40
           1      0.273     0.469     0.345        32
           2      0.125     0.033     0.053        30
           3      0.000     0.000     0.000        14

    accuracy                          0.276       116
   macro avg      0.178     0.226     0.187       116
weighted avg      0.216     0.276     0.230       116

Pred distribution: [51 55  8  2]
True distribution: [40 32 30 14]
Train  loss=1.4437 acc=0.3914 f1=0.3646 | Val loss=1.9035 acc=0.2759 f1=0.1873
Early stopping due to no improvement in 7 epochs.
Restored best weights for fold 4 (F1=0.2337)


# tf_efficientnetv2_s.in21k

In [7]:
def create_model_tf_efficientnetv2_s(pretrained: bool = True) -> nn.Module:
    model = timm.create_model(
        PRETRAINED_MODEL,
        pretrained=pretrained,
        num_classes=N_CLASSES,
        in_chans=4,
        drop_rate=0.3,        # Dropout
        drop_path_rate=0.1    # Stochastic depth
    ).to(device)
    return model

if MODEL_TO_USE == PreTrainedArchitectures.EFFICIENTNETV2_S:
    N_FOLDS = data.num_K_folds
    IMAGE_SIZE = data.image_size
    BATCH_SIZE = 4
    PRETRAINED_MODEL = MODEL_TO_USE.value
    N_CLASSES = 4 # number of classes in the dataset (labels)
    N_WORKERS = os.cpu_count() // 2 if os.cpu_count() else 4
    EPOCHS_STAGE1 = 10
    EPOCHS_STAGE2 = 15

    for fold in range(N_FOLDS):
        print(f"\n========== Fold {fold} ==========")

        train_df_split = train_df[train_df["fold"] != fold].reset_index(drop=True)
        val_df_split   = train_df[train_df["fold"] == fold].reset_index(drop=True)

        train_dataset = HistologyDataset(
            df=train_df_split,
            image_size=IMAGE_SIZE,
            is_train=True,
            use_mask_crop=True
        )
        val_dataset = HistologyDataset(
            df=val_df_split,
            image_size=IMAGE_SIZE,
            is_train=False,   # False to disable augmentations
            use_mask_crop=True
        )

        sampler = make_weighted_sampler(train_df_split)
        train_loader = DataLoader(
            train_dataset,
            batch_size=BATCH_SIZE,
            sampler=sampler,
            num_workers=N_WORKERS,
            pin_memory=cuda_is_available
        )
        val_loader   = DataLoader(
            val_dataset,
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=N_WORKERS,
            pin_memory=cuda_is_available
        )

        # --- create fresh model for this fold ---
        model = create_model_tf_efficientnetv2_s()

        # --- Stage 1: freeze backbone, train classifier head ---
        print("\n--- Stage 1: Training classifier head ---")

        # --- 1.1. freeze feature extractor layers ---
        for param in model.parameters():
            param.requires_grad = False

        # 2) unfreeze classifier head (EffNetV2 uses .classifier)
        for param in model.classifier.parameters():
            param.requires_grad = True

        # --- 1.2. define loss, optimizer, scheduler ---
        criterion = nn.CrossEntropyLoss()
        head_params = [p for p in model.parameters() if p.requires_grad]
        optimizer = AdamW(head_params, lr=1e-3, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS_STAGE1)

        # --- 1.3. train for several epochs ---
        best_f1 = 0.0
        best_state = None
        for epoch in range(1, EPOCHS_STAGE1+1):
            print(f"\nEpoch {epoch}/{EPOCHS_STAGE1}")
            train_loss, train_acc, train_f1 = train_one_epoch(
                model, train_loader, optimizer, criterion, device
            )
            val_loss, val_acc, val_f1 = validate(
                model, val_loader, criterion, device
            )
            scheduler.step()
            print(
                f"Train  loss={train_loss:.4f} acc={train_acc:.4f} f1={train_f1:.4f} | "
                f"Val loss={val_loss:.4f} acc={val_acc:.4f} f1={val_f1:.4f}"
            )
            if val_f1 > best_f1:
                best_f1 = val_f1
                best_state = deepcopy(model.state_dict())
                torch.save(best_state, f"best_effv2_stage1_fold{fold}_f1_{val_f1:.4f}.pth")
                print(f"  🔥 New best F1: {best_f1:.4f} – model saved.")

        if best_state is not None:
            model.load_state_dict(best_state)
            print(f"Restored best Stage 1 weights for fold {fold} (F1={best_f1:.4f})")

        # --- Stage 2: unfreeze whole model, fine-tune ---
        print("\n--- Stage 2: Fine-tuning entire model ---")

        # --- 2.1. unfreeze entire model ---
        for param in model.parameters():
            param.requires_grad = True

        # --- 2.2. define loss, optimizer, scheduler ---
        optimizer = AdamW(model.parameters(), lr=5e-5, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS_STAGE2)

        # --- 2.3. mild class weights ---
        class_counts_np = train_df["label_idx"].value_counts().sort_index().values
        class_counts = torch.tensor(class_counts_np, dtype=torch.float32) # 445 LumB, 414 LumA, 397 Her2, 156 TN
        class_weights = (class_counts.sum() / class_counts)
        class_weights = class_weights / class_weights.mean()
        # criterion = FocalLoss(alpha=class_weights, gamma=2.0)
        criterion = nn.CrossEntropyLoss(weight=class_weights.to(device), label_smoothing=0.1)

        # --- 2.4. train for several epochs ---
        best_f1 = 0.0
        best_state = None

        for epoch in range(1, EPOCHS_STAGE2 + 1):
            print(f"\nEpoch {epoch}/{EPOCHS_STAGE2}")
            train_loss, train_acc, train_f1 = train_one_epoch(
                model, train_loader, optimizer, criterion, device
            )
            val_loss, val_acc, val_f1 = validate(
                model, val_loader, criterion, device
            )
            scheduler.step()
            print(
                f"Train  loss={train_loss:.4f} acc={train_acc:.4f} f1={train_f1:.4f} | "
                f"Val loss={val_loss:.4f} acc={val_acc:.4f} f1={val_f1:.4f}"
            )
            if val_f1 > best_f1:
                best_f1 = val_f1
                best_f1_per_fold[fold] = best_f1
                best_state = deepcopy(model.state_dict())
                torch.save(best_state, f"best_effv2_stage2_fold{fold}_f1_{val_f1:.4f}.pth")
                print(f"  🔥 New best F1: {best_f1:.4f} – model saved.")

        if best_state is not None:
            model.load_state_dict(best_state)   # restore best val-F1 weights

        # --- save model for this fold ---
        torch.save(model.state_dict(), f"effv2_s_fold{fold}.pth")

# convnext_tiny

In [8]:
def create_model_convnext(pretrained: bool = True) -> nn.Module:
    model = timm.create_model(
        PRETRAINED_MODEL,
        pretrained=pretrained,
        num_classes=N_CLASSES,
        in_chans=4,
        drop_rate=0.3,        # Dropout
        drop_path_rate=0.1    # Stochastic depth
    ).to(device)
    return model

if MODEL_TO_USE == PreTrainedArchitectures.CONVNEXT_TINY:
    N_FOLDS = data.num_K_folds
    IMAGE_SIZE = data.image_size
    BATCH_SIZE = 4
    PRETRAINED_MODEL = MODEL_TO_USE.value
    N_CLASSES = 4  # number of classes in the dataset (labels)
    N_WORKERS = os.cpu_count() // 2 if os.cpu_count() else 4
    EPOCHS_STAGE1 = 8
    EPOCHS_STAGE2 = 12

    for fold in range(N_FOLDS):
        print(f"\n========== Fold {fold} ==========")

        train_df_split = train_df[train_df["fold"] != fold].reset_index(drop=True)
        val_df_split   = train_df[train_df["fold"] == fold].reset_index(drop=True)

        train_dataset = HistologyDataset(
            df=train_df_split,
            image_size=IMAGE_SIZE,
            is_train=True,
            use_mask_crop=True
        )
        val_dataset = HistologyDataset(
            df=val_df_split,
            image_size=IMAGE_SIZE,
            is_train=False,   # Disable augmentations
            use_mask_crop=True
        )

        sampler = make_weighted_sampler(train_df_split)
        train_loader = DataLoader(
            train_dataset,
            batch_size=BATCH_SIZE,
            sampler=sampler,
            num_workers=N_WORKERS,
            pin_memory=cuda_is_available
        )
        val_loader = DataLoader(
            val_dataset,
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=N_WORKERS,
            pin_memory=cuda_is_available
        )

        # --- create fresh model for this fold ---
        model = create_model_convnext()

        # --- Stage 1: freeze backbone, train classifier HEAD (ConvNeXt) ---
        print("\n--- Stage 1: Training classifier head (ConvNeXt-Tiny) ---")

        # --- 1.1. freeze feature extractor layers ---
        for p in model.parameters():
            p.requires_grad = False

        # --- 1.1. unfreeze only the classifier head (ConvNeXt uses .head) ---
        for p in model.head.parameters():
            p.requires_grad = True

        # --- 1.2. define loss, optimizer, scheduler ---
        criterion = nn.CrossEntropyLoss()
        head_params = [p for p in model.parameters() if p.requires_grad]
        optimizer = AdamW(head_params, lr=1e-3, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS_STAGE1)

        # --- 1.3. train for several epochs ---
        best_f1 = 0.0
        best_state = None
        for epoch in range(1, EPOCHS_STAGE1 + 1):
            print(f"\nEpoch {epoch}/{EPOCHS_STAGE1}")
            train_loss, train_acc, train_f1 = train_one_epoch(
                model, train_loader, optimizer, criterion, device
            )
            val_loss, val_acc, val_f1 = validate(
                model, val_loader, criterion, device
            )
            scheduler.step()
            print(
                f"Train  loss={train_loss:.4f} acc={train_acc:.4f} f1={train_f1:.4f} | "
                f"Val loss={val_loss:.4f} acc={val_acc:.4f} f1={val_f1:.4f}"
            )
            if val_f1 > best_f1:
                best_f1 = val_f1
                best_state = deepcopy(model.state_dict())
                torch.save(best_state, f"best_convnext_stage1_fold{fold}_f1_{val_f1:.4f}.pth")
                print(f"  🔥 New best F1: {best_f1:.4f} – model saved.")

        if best_state is not None:
            model.load_state_dict(best_state)
            print(f"Restored best Stage 1 weights for fold {fold} (F1={best_f1:.4f})")

        # --- Stage 2: unfreeze whole model, fine-tune ---
        print("\n--- Stage 2: Fine-tuning entire model (ConvNeXt-Tiny) ---")

        # --- 2.1. unfreeze entire model ---
        for p in model.parameters():
            p.requires_grad = True

        # --- 2.2. define loss, optimizer, scheduler ---
        optimizer = AdamW(model.parameters(), lr=5e-5, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS_STAGE2)

        # --- 2.3. mild class weights ---
        class_counts_np = train_df["label_idx"].value_counts().sort_index().values
        class_counts = torch.tensor(class_counts_np, dtype=torch.float32)  # 445 LumB, 414 LumA, 397 Her2, 156 TN
        class_weights = (class_counts.sum() / class_counts)
        class_weights = class_weights / class_weights.mean()
        # criterion = FocalLoss(alpha=class_weights, gamma=2.0)
        criterion = nn.CrossEntropyLoss(weight=class_weights.to(device), label_smoothing=0.1)

        # --- 2.4. train for several epochs ---
        best_f1 = 0.0
        best_state = None
        for epoch in range(1, EPOCHS_STAGE2 + 1):
            print(f"\nEpoch {epoch}/{EPOCHS_STAGE2}")
            train_loss, train_acc, train_f1 = train_one_epoch(
                model, train_loader, optimizer, criterion, device
            )
            val_loss, val_acc, val_f1 = validate(
                model, val_loader, criterion, device
            )
            scheduler.step()
            print(
                f"Train  loss={train_loss:.4f} acc={train_acc:.4f} f1={train_f1:.4f} | "
                f"Val loss={val_loss:.4f} acc={val_acc:.4f} f1={val_f1:.4f}"
            )
            if val_f1 > best_f1:
                best_f1 = val_f1
                best_f1_per_fold[fold] = best_f1
                best_state = deepcopy(model.state_dict())
                torch.save(best_state, f"best_convnext_stage2_fold{fold}_f1_{val_f1:.4f}.pth")
                print(f"  🔥 New best F1: {best_f1:.4f} – model saved.")

        if best_state is not None:
            model.load_state_dict(best_state)  # restore best val-F1 weights

        # --- save model for this fold ---
        torch.save(model.state_dict(), f"convnext_tiny_fold{fold}.pth")

# Model Inference with 5-Fold Ensembling

In [9]:
prefix_filename = "effv2_s" if MODEL_TO_USE == PreTrainedArchitectures.EFFICIENTNETV2_S else "convnext_tiny" if MODEL_TO_USE == PreTrainedArchitectures.CONVNEXT_TINY else "effb0" if MODEL_TO_USE == PreTrainedArchitectures.EFFICIENTNET_B0 else "effb1" if MODEL_TO_USE == PreTrainedArchitectures.EFFICIENTNET_B1 else "tf_effb1_ns"

FOLD_VAL_F1 = f"fold_val_f1_{prefix_filename}.json"

# Save best F1 per fold to JSON
with open(FOLD_VAL_F1, "w") as f:
    json.dump(best_f1_per_fold, f, indent=2)

In [10]:
def predict_loader_with_tta(model, loader, device):
    model.eval()
    all_probs = []
    all_targets = []

    with torch.no_grad():
        for imgs, labels in tqdm(loader):
        # for imgs, labels in loader:  # note: here we have labels, not sample_index
            imgs = imgs.to(device)  # if batch_size=1
            tta_imgs = apply_tta(imgs[0])         # same apply_tta as for test

            probs_sum = 0
            for aug in tta_imgs:
                aug = aug.unsqueeze(0).to(device)
                logits = model(aug)
                probs = softmax(logits, dim=1)
                probs_sum += probs[0].cpu().numpy()

            avg_probs = probs_sum / len(tta_imgs)
            all_probs.append(avg_probs)
            all_targets.append(labels.item())

    all_probs = np.vstack(all_probs)
    all_targets = np.array(all_targets)
    pred_indices = all_probs.argmax(axis=1)

    macro_f1 = f1_score(all_targets, pred_indices, average="macro")
    return macro_f1

def predict_loader_no_tta(model, loader, device):
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for imgs, labels in tqdm(loader):
        # for imgs, labels in loader:
            imgs = imgs.to(device)
            logits = model(imgs)
            preds.append(logits.argmax(1).cpu().item())
            targets.append(labels.item())
    return f1_score(targets, preds, average="macro")

fold_f1s = []
fold_f1s_no_tta = []

for fold in range(data.num_K_folds):
    print(f"OOF eval for fold {fold}")

    # build val_df_split for that fold
    val_df_split = train_df[train_df["fold"] == fold].reset_index(drop=True)
    val_dataset = HistologyDataset(
        df=val_df_split,
        image_size=data.image_size,
        is_train=False,   # Disable augmentations
        use_mask_crop=True,
        apply_artifact_augs=False
    )
    val_loader  = DataLoader(val_dataset, batch_size=1, shuffle=False,
                             num_workers=N_WORKERS, pin_memory=cuda_is_available)

    if MODEL_TO_USE == PreTrainedArchitectures.EFFICIENTNETV2_S:
        model = create_model_tf_efficientnetv2_s(pretrained=False)
    elif MODEL_TO_USE == PreTrainedArchitectures.CONVNEXT_TINY:
        model = create_model_convnext(pretrained=False)
    elif MODEL_TO_USE == PreTrainedArchitectures.EFFICIENTNET_B1_NS:
        model = create_efficientnet_b1_ns_model(pretrained=False)
    else:
        model = create_efficientnet_b0_model(pretrained=False)
    model.load_state_dict(torch.load(f"{prefix_filename}_fold{fold}.pth", map_location=device))

    f1 = predict_loader_with_tta(model, val_loader, device)
    fold_f1s.append(f1)
    print("Fold F1 (OOF, with TTA):", f1)
    f1_no_tta = predict_loader_no_tta(model, val_loader, device)
    fold_f1s_no_tta.append(f1_no_tta)
    print("Fold F1 (OOF, no TTA):", f1_no_tta)

print("Mean OOF F1:", np.mean(fold_f1s))
print("Mean OOF F1 (no TTA):", np.mean(fold_f1s_no_tta))


OOF eval for fold 0


100%|██████████| 117/117 [00:20<00:00,  5.63it/s]


Fold F1 (OOF, with TTA): 0.29957543193578895


100%|██████████| 117/117 [00:03<00:00, 31.26it/s]


Fold F1 (OOF, no TTA): 0.33346171802054153
OOF eval for fold 1


...

Fold F1 (OOF, no TTA): 0.21048819637474064
OOF eval for fold 4


100%|██████████| 116/116 [00:20<00:00,  5.64it/s]


Fold F1 (OOF, with TTA): 0.23544864226682408


100%|██████████| 116/116 [00:03<00:00, 29.94it/s]

Fold F1 (OOF, no TTA): 0.23373466887879304
Mean OOF F1: 0.21768503155232538
Mean OOF F1 (no TTA): 0.24173553753428698


In [11]:
########################################################
# ===== Inference with TTA and 5-Fold Ensembling ===== #
########################################################
all_fold_probs = []
all_sample_indices = None

test_dataset = HistologyDataset(
    df=test_df,
    image_size=data.image_size,
    is_train=False,   # deterministic, returns (img, sample_index)
    use_mask_crop=True,
    patch_mode=False,
    apply_artifact_augs=False
)
test_loader = DataLoader(
    test_dataset,
    batch_size=1,               # per-image TTA
    shuffle=False,
    num_workers=N_WORKERS,
    pin_memory=cuda_is_available
)

if os.path.exists(FOLD_VAL_F1):
    with open(FOLD_VAL_F1, "r") as f:
        best_f1_per_fold = json.load(f)
    val_f1_per_fold = np.array([best_f1_per_fold[str(k)] for k in range(data.num_K_folds)])
    # Normalize to get weights that sum to 1
    # fold_weights = val_f1_per_fold / val_f1_per_fold.sum()
    fold_weights = np.ones(data.num_K_folds, dtype=np.float32) / data.num_K_folds
else:
    # fallback: uniform weights if metrics are missing
    print('Warning: fold validation F1 scores not found, using uniform weights.')
    fold_weights = np.ones(data.num_K_folds, dtype=np.float32) / data.num_K_folds

print("Fold weights:", fold_weights)

# -----------------------------
# 2) Accumulate weighted probs
# -----------------------------
all_probs = None
all_sample_indices = None

for fold in range(data.num_K_folds):
    print(f"Inference with fold {fold} model (weight={fold_weights[fold]:.3f})")

    if MODEL_TO_USE == PreTrainedArchitectures.EFFICIENTNETV2_S:
        model = create_model_tf_efficientnetv2_s(pretrained=False)
    elif MODEL_TO_USE == PreTrainedArchitectures.CONVNEXT_TINY:
        model = create_model_convnext(pretrained=False)
    elif MODEL_TO_USE == PreTrainedArchitectures.EFFICIENTNET_B1 or MODEL_TO_USE == PreTrainedArchitectures.EFFICIENTNET_B0:
        model = create_efficientnet_b0_model(pretrained=False)
    else:
        model = create_efficientnet_b1_ns_model(pretrained=False)

    state_dict = torch.load(f"{prefix_filename}_fold{fold}.pth", map_location=device)
    model.load_state_dict(state_dict)
    model.eval()

    fold_probs = []
    sample_indices_list = []

    with torch.no_grad():
        for img_tensor, sample_idx in tqdm(test_loader):
        # for img_tensor, sample_idx in test_loader:
            # img_tensor: [1, 4, H, W]  (RGB+mask)
            img_tensor = img_tensor.squeeze(0).to(device)  # [4, H, W]

            # -------- TTA: mask-based multi-crop + simple flips --------
            USE_MASK_TTA = False
            if USE_MASK_TTA:
                tta_tensors = apply_mask_multicrop_tta(img_tensor, crop_size=data.image_size, n_crops=2)
            else:
                tta_tensors = apply_tta_4ch_safe(img_tensor)

            logits_sum = 0
            for aug in tta_tensors:
                aug = aug.unsqueeze(0).to(device)
                logits_sum += model(aug)[0].detach().cpu().numpy()

            avg_logits = logits_sum / len(tta_tensors)
            avg_probs = softmax(torch.tensor(avg_logits), dim=0).numpy()


            # accumulate probability predictions
            # probs_sum = 0.0
            # for aug_img in tta_tensors:
            #     aug_img = aug_img.unsqueeze(0).to(device)  # [1, 4, H, W]
            #     with torch.no_grad():
            #         logits = model(aug_img)
            #         probs = softmax(logits, dim=1)  # [1, N_CLASSES]
            #     probs_sum += probs[0].cpu().numpy()
            #
            # # average across TTA views
            # avg_probs = probs_sum / len(tta_tensors)

            fold_probs.append(avg_probs)

            # collect sample indices only once
            if all_sample_indices is None:
                sample_indices_list.append(sample_idx[0])

    fold_probs = np.vstack(fold_probs)  # [N_test, N_CLASSES]

    # initialize global probs
    if all_probs is None:
        all_probs = np.zeros_like(fold_probs, dtype=np.float32)

     # weighted accumulation
    all_probs += fold_weights[fold] * fold_probs

    if all_sample_indices is None:
        all_sample_indices = sample_indices_list

# -----------------------------
# 3) Final predictions
# -----------------------------
pred_indices = all_probs.argmax(axis=1)
pred_labels = [idx2label[int(i)] for i in pred_indices]

sample_index_with_ext = [
    f"{si}.png" if not si.endswith(".png") else si
    for si in all_sample_indices
]

submission_df = pd.DataFrame({
    "sample_index": sample_index_with_ext,
    "label": pred_labels
})
submission_df.to_csv(f"submission_5fold_tta_{prefix_filename}.csv", index=False)

print(f"Saved submission_5fold_tta_{prefix_filename}.csv")

Fold weights: [0.2 0.2 0.2 0.2 0.2]
Inference with fold 0 model (weight=0.200)


100%|██████████| 477/477 [00:27<00:00, 17.33it/s]


Inference with fold 1 model (weight=0.200)


100%|██████████| 477/477 [00:27<00:00, 17.33it/s]


Inference with fold 2 model (weight=0.200)


...

Inference with fold 3 model (weight=0.200)


100%|██████████| 477/477 [00:27<00:00, 17.19it/s]


Inference with fold 4 model (weight=0.200)


100%|██████████| 477/477 [00:27<00:00, 17.29it/s]

Saved submission_5fold_tta_tf_effb1_ns.csv
